In [ ]:
from genQC.imports import *
import genQC.util as util
import genQC.platform.qcircuit_dataset_construction as data_const
import genQC.inference.infer_srv as infer_srv
import genQC.dataset.dataset_helper as dahe
from genQC.platform.simulation.qcircuit_sim import instruction_name_to_qiskit_gate
from genQC.pipeline.diffusion_pipeline import DiffusionPipeline
from genQC.pipeline.diffusion_pipeline_special import DiffusionPipeline_Compilation
from genQC.dataset.qc_dataset import Qc_Config_Dataset
from genQC.dataset.mixed_cached_qc_dataset import Mixed_Cached_OpenClip_Dataset
from genQC.platform.qcircuit_metrics import Unitary_FrobeniusNorm

In [ ]:
device = util.infer_torch_device()  # use cuda if we can, cpu is much slower
util.MemoryCleaner.purge_mem()      # clean existing memory alloc

## Setup and load

In [ ]:
model_path = "./saves/qc_unet_config_Compilation_3_qubit/"
pipeline = DiffusionPipeline_Compilation.from_config_file_initial(model_path, device)

pipeline.guidance_sample_mode = "rescaled"
pipeline.scheduler.set_timesteps(20) 

print("Trained with gates:", pipeline.gate_pool)

## Fine-tune dataset

### Sampling random circuits

In [ ]:
# settings for random circuit sampling
random_samples = int(1e2)                 # how many rnd qcs we sample, here small number to speed up example
num_of_qubits  = 3
min_gates      = 2
max_gates      = 12
gate_pool      = [instruction_name_to_qiskit_gate(gate) for gate in pipeline.gate_pool] 
optimized      = True                     # if qiskit optimizer is used

x,y,U,params = data_const.gen_compilation_rndGates_dataset_param(samples=random_samples, num_of_qubits=num_of_qubits, min_gates=min_gates, max_gates=max_gates, 
                                gate_pool=gate_pool)

y = np.array(y)


In [ ]:
print(params.shape)
print(U.shape)
print(x.shape)
print(y[1])

In [ ]:
print(f"Example circuit with prompt: \n{y[33]} \n{x[33]} \n{U[33]} \n\n{params[33]}")

### Create a basic dataset

In [ ]:
# meta-data of dataset
paras = {}
paras["store_dict"]     = {'x':'tensor', 'y':'numpy', 'params':'tensor', 'U':'tensor'}   #what is in the datset, with type
paras["optimized"]      = optimized    
paras["dataset_to_gpu"] = True if device=="cuda" else False
paras["random_samples"] = random_samples
paras["num_of_qubits"]  = num_of_qubits
paras["min_gates"]      = min_gates
paras["max_gates"]      = max_gates
paras["gate_pool"]      = pipeline.gate_pool

Make sure our dataset has no duplicates and shuffle it:

In [ ]:
# x, y = dahe.uniquify_tensor_dataset(x, y)
# assert x.shape[0] == x.unique(dim=0).shape[0]    # check if no duplicates

# x, y = dahe.shuffle_tensor_dataset(x, y)
from genQC.platform.qcircuit_dataset_construction import decode_circuit
print(x[33])
print()
print(params[33])
print(U[33])

qc = decode_circuit(x[33], gate_pool, params_tensor=params[33])
display(qc.draw("mpl", plot_barriers=False))



Now create the `Qc_Config_Dataset` object:

In [ ]:
qc_Config_Dataset = Qc_Config_Dataset(store_device=device, **paras)
qc_Config_Dataset.x = x
qc_Config_Dataset.y = y
qc_Config_Dataset.U = U
qc_Config_Dataset.params = params
qc_Config_Dataset.dataset_to_gpu = True if device.type=="cuda" else False



qc_Config_Dataset.plot_example()

### Create a cached (mixed) dataset

To speed up training we can cache the CLIP embeddings of the `y` dataset labels before we start fitting. We provide the `Cached_OpenClip_Dataset` object for this. Here we use a further extension, the `Mixed_Cached_OpenClip_Dataset`. It has advanced methods to handle padding and combining different task (e.g. compile and SRV) or different number of qubit datasets together (as explained in the appendix of the paper). We use it here to automatically cut and pad our 9 qubit circuits to the longest circuit within one batch.

See if the pipeline has already a padding token specified, else define one.

In [ ]:
try:    pad_constant = pipeline.params_config("")["add_config"]["dataset"]["params"]["pad_constant"]  #can NOT be 0 (empty token)! and not any other gate!
except: pad_constant = len(qc_Config_Dataset.gate_pool)+1
    
print(f"{pad_constant=}")

In [ ]:
dataset_list = [qc_Config_Dataset]                  # what datasets to combine

parameters   = asdict(qc_Config_Dataset.params_config)
parameters["num_down_scales"] = 3                   # defined by the down-scale layers of the UNet
# print(parameters)

mixed_dataset = Mixed_Cached_OpenClip_Dataset.from_datasets(dataset_list,                 
                 balance_maxes=[1e8],          # what the maximum prompt (y) balance limit is, can be used to balance SRVs for different qubit numbers                                      
                 pad_constant=pad_constant,
                 device=device, 
                 bucket_batch_size=-1,         # if we use bucket padding
                 max_samples=[1e8],            # if we want to limit the sizes of the dataset_list 
                 **parameters)

Finally, we can create the dataloader used by the `DiffusionPipeline.fit()` funtion. This also caches all our prompts.

In [ ]:


dataloaders = mixed_dataset.get_dataloaders(batch_size=16, text_encoder=pipeline.text_encoder.to(device), y_on_cpu=False)  # you can set y_on_cpu=True if you run out of device mem

## Fine-tune

We have the dataloader object created and can start fine-tuning. Note, we just use all the diffusion scheduler parameters from the pre-trained config we loaded. 

In [ ]:
pipeline.add_config["dataset"] = mixed_dataset.get_config()   # add meta-data of dataset to save it with pipeline
pipeline.compile(torch.optim.Adam, nn.MSELoss)
# print(mixed_dataset.get_config())

In [ ]:
epochs = 30     # how many epochs we train on our 9bit dataset
lr     = 3e-4    # learn rate

sched = functools.partial(torch.optim.lr_scheduler.OneCycleLR, max_lr=lr, total_steps=epochs*len(dataloaders.train))
pipeline.fit(epochs, dataloaders, lr=lr, lr_sched=sched)

In [ ]:
store_dir = f"./saves/qc_params-new/"
pipeline.store_pipeline(config_path=store_dir, save_path=store_dir)